# Unificar y limpiar la base de rasgos funcionales de Selva Viva

Parte de 2 archivos:
- **`04__Rasgos_SelvaViva.xlsx`** (hoja `Hoja 1`): **base principal**, 48 árboles, muestreo de **febrero de 2025**.
- **`Base_SelvaViva_v2.xlsx`**, hoja **`Rasgos`** (no la hoja `SelvaViva2024`, que es el listado completo de dinámica): 14 árboles, muestreo de **septiembre de 2025**.

Son dos campañas de muestreo en fechas distintas, casi sin superposición de árboles (solo 1 árbol se repite en ambas). El proceso es el mismo que se usó para Galeras: estandarizar columnas, cruzar identificadores, unir, limpiar, calcular rasgos, revisar rangos y conservar el resaltado amarillo original.

**A diferencia de Galeras, aquí SÍ hay peso fresco de hoja** (`Leaf fresh weight (g)`) en ambos archivos, así que esta vez **sí se puede calcular LDMC**. `force to punch` sigue sin poder calcularse (no hay dato crudo de punzón en ninguna de las 2 bases).

## 1. Librerías

In [144]:
import pandas as pd
import numpy as np
import re
import openpyxl
from openpyxl.styles import PatternFill

pd.set_option('display.max_columns', None)

## 2. Extraer las celdas resaltadas en amarillo

Igual que en Galeras: se recorre cada archivo con `openpyxl` y se guarda qué celdas tienen relleno amarillo puro (`FFFFFF00`, tipo `rgb`). La hoja `Rasgos` de `Base_SelvaViva_v2.xlsx` solo tiene colores de **tema** (bandas de color para lectura, no marcas de revisión), así que no aporta resaltados reales; el amarillo real está solo en `04`.

In [145]:
def extraer_resaltado_amarillo(path, sheet_name):
    wb = openpyxl.load_workbook(path)
    ws = wb[sheet_name]
    headers = [c.value for c in next(ws.iter_rows(min_row=1, max_row=1))]
    resaltado = {}
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            fill = cell.fill
            if fill and fill.fgColor and fill.fgColor.type == 'rgb' and fill.fgColor.rgb == 'FFFFFF00':
                col_name = headers[cell.column - 1]
                resaltado.setdefault(cell.row - 2, set()).add(col_name)
    return resaltado

resaltado_04 = extraer_resaltado_amarillo('04. Rasgos_SelvaViva.xlsx', 'Hoja 1')
resaltado_v2 = extraer_resaltado_amarillo('Base_SelvaViva v2.xlsx', 'Rasgos')
print('Filas con amarillo en 04:', len(resaltado_04))
print('Filas con amarillo en v2 Rasgos:', len(resaltado_v2))

Filas con amarillo en 04: 5
Filas con amarillo en v2 Rasgos: 0


## 3. Función auxiliar de limpieza numérica

In [146]:
def a_numero(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan

## 4. Cargar y estandarizar `04` (base principal, feb-2025)

`Revisado Karina fotos` se conserva como `nota_revision_campo`: es una columna de comentarios de campo ("Revisar Jurgen", "revisar campo", etc.) que se traslada directamente al `QC_flag` más abajo — son observaciones que el propio equipo de campo ya había dejado. El prefijo de parcela se homogeniza de `SV_XX` a `SEV_XX` para que coincida con la convención usada en la base de dinámica.

In [147]:
df04 = pd.read_excel('04. Rasgos_SelvaViva.xlsx', sheet_name='Hoja 1')

ren04 = {
    'Site': 'Site', 'Plot_code': 'PlotID_raw', 'Sub': 'Subplot', 'altitude': 'altitude_m',
    'Date ': 'sampling_date', 'TreeID_2025': 'new_tree_ID_2025',
    'Revisado Karina fotos': 'nota_revision_campo',
    'Family': 'family', 'genus': 'genus', 'specie': 'species',
    'leaf 1, thickn 1': 'leaf1_thickness1', 'leaf 1, thickn 2': 'leaf1_thickness2', 'leaf 1, thickn 3': 'leaf1_thickness3',
    'leaf 2, thickn 1': 'leaf2_thickness1', 'leaf 2, thickn 2': 'leaf2_thickness2', 'leaf 2, thickn 3': 'leaf2_thickness3',
    'leaf 3, thickn 1': 'leaf3_thickness1', 'leaf 3, thickn 2': 'leaf3_thickness2', 'leaf 3, thickn 3': 'leaf3_thickness3',
    '# leaves': 'n_leaves', 'Leaf fresh weight (g)': 'leaf_fresh_weight_g', 'Leaf dry weight (g)': 'leaf_dry_weight_g',
    'Leaf area (cm²)': 'leaf_area_cm2', 'SLA': 'SLA_original', 'Comment': 'comment',
    'wood characteristics': 'wood_characteristics', 'crust thickness': 'crust_thickness_cm',
    'wet lenght cm': 'wet_length_cm_raw', 'wet wood weight': 'wet_wood_weight_g', 'dry wood weight': 'dry_wood_weight_g',
}
df04 = df04.rename(columns=ren04)
df04 = df04.drop(columns=['#', 'mean_thickness_(mm)'], errors='ignore')
df04['PlotID'] = df04['PlotID_raw'].str.replace('^SV_', 'SEV_', regex=True)
df04['Plot'] = df04['PlotID'].str.extract(r'(\d+)').astype(float)
df04['treeID'] = np.nan  # se completa con el cruce mas abajo
df04['source_file'] = '04__Rasgos_SelvaViva.xlsx (principal, feb-2025)'
df04['_orig_row'] = range(len(df04))
df04.shape

(48, 35)

## 5. Cargar y estandarizar la hoja `Rasgos` de `Base_SelvaViva_v2.xlsx` (sept-2025)

Esta hoja trae, a diferencia de `04`, tanto el `treeID` viejo como el `new ID 2024` — eso permite construir el cruce de identificadores. También trae columnas de censo (`dbh 1`, `dbh 2`, etc.) que se guardan por si son útiles más adelante, aunque no son el foco de este notebook.

In [148]:
dfv2 = pd.read_excel('Base_SelvaViva v2.xlsx', sheet_name='Rasgos')

renv2 = {
    'site': 'Site', 'PlotID': 'PlotID_raw', 'treeID': 'treeID', 'new ID 2024': 'new_tree_ID_2025',
    'family': 'family', 'genus': 'genus', 'species': 'species',
    'date census 1': 'date_census1', 'dbh 1': 'dbh_census1',
    'date census 2': 'date_census2', 'dbh 2': 'dbh_census2', 'tree height 2011 (m)': 'height_census2_m',
    'comment': 'comment_dinamica', 'JH herbarium collections': 'JH_herbarium',
    'sampling date_leaves 2025': 'sampling_date', 'comp. leaf': 'compound_leaf_note',
    'No. leaves': 'n_leaves', 'Leaf fresh weight (g)': 'leaf_fresh_weight_g', 'Leaf dry weight (g)': 'leaf_dry_weight_g',
    'Comments': 'comment',
    'leaf 1 thickness 1': 'leaf1_thickness1', 'leaf 1 thickness 2': 'leaf1_thickness2',
    'leaf 2 thickness 1': 'leaf2_thickness1', 'leaf 2 thickness 2': 'leaf2_thickness2',
    'leaf 3 thickness 1': 'leaf3_thickness1', 'leaf 3 thickness 2': 'leaf3_thickness2',
    'crust thickness': 'crust_thickness_cm', 'wet lenght cm': 'wet_length_cm_raw',
    'wet wood weight': 'wet_wood_weight_g', 'dry wood weight': 'dry_wood_weight_g',
    'comment wood': 'comment_wood',
}
dfv2 = dfv2.rename(columns=renv2)
dfv2 = dfv2.drop(columns=['Individual tree census 1', 'new census 2', 'dead census 2', 'dendrometer',
                          'leaf sample 2025', 'wood sample 2025', 'sampling leaf thickness'], errors='ignore')
dfv2['PlotID'] = dfv2['PlotID_raw']  # ya viene como SEV_XX
dfv2['Plot'] = dfv2['PlotID'].str.extract(r'(\d+)').astype(float)
dfv2['source_file'] = 'Base_SelvaViva_v2.xlsx / hoja Rasgos (sept-2025)'
dfv2['_orig_row'] = range(len(dfv2))
dfv2.shape

(14, 35)

## 6. Cruzar `treeID` viejo con `new_tree_ID_2025`

La hoja `Rasgos` de `v2` trae ambos IDs para cada árbol, así que sirve de puente para completar el `treeID` viejo en `04` (que solo trae el ID nuevo).

In [149]:
crosswalk = (dfv2[dfv2['new_tree_ID_2025'].notna()]
             .drop_duplicates('new_tree_ID_2025')
             .set_index('new_tree_ID_2025')['treeID'])
df04['treeID'] = df04['new_tree_ID_2025'].map(crosswalk)

# Añadir LA

## 7. Unificar las 2 campañas

Solo **1 árbol** (`new_tree_ID_2025 = 3658`, plot `SEV_70`) fue muestreado en ambas campañas — se conservan ambas filas (no se deduplican, porque son mediciones legítimas en fechas distintas) y se marcan para que compares los valores. Las notas de campo (`Revisar Jurgen`, `revisar campo`, etc.) que ya traía `04` se trasladan también al `QC_flag`.

In [150]:
unificado = pd.concat([df04, dfv2], ignore_index=True, sort=False)
unificado['QC_flag'] = ''

repetidos = unificado[unificado.duplicated('new_tree_ID_2025', keep=False) & unificado['new_tree_ID_2025'].notna()]
print('Arboles muestreados en ambas campanas:', repetidos['new_tree_ID_2025'].nunique())

unificado.loc[unificado['new_tree_ID_2025'].isin(repetidos['new_tree_ID_2025']), 'QC_flag'] += \
    'Arbol muestreado en las 2 campanas (fechas distintas); comparar valores; '

tiene_nota = unificado['nota_revision_campo'].notna() & ~unificado['nota_revision_campo'].isin([0, 1])
unificado.loc[tiene_nota, 'QC_flag'] += (
    'Nota de campo: ' + unificado.loc[tiene_nota, 'nota_revision_campo'].astype(str) + '; ')

print('Base unificada:', unificado.shape)

Arboles muestreados en ambas campanas: 1
Base unificada: (62, 45)


In [151]:
LA = pd.read_excel("SV_2025_LA.xlsx")
unificado = pd.merge(unificado,LA, how="left",right_on="TreeID",left_on="new_tree_ID_2025")
unificado["leaf_area_cm2"] = unificado["leaf_area_cm2"].combine_first(
    unificado["SUM Leaf area (cm²)"])

## 8. Limpiar valores numéricos y grosor de hoja

Se detectó una celda de grosor de hoja con valor `17.00` (plot `SEV_74`, árbol 8839) — claramente un error de captura (falta el punto decimal, el resto de mediciones de esa fila están entre 0.13–0.19 mm). Se excluye del promedio y se marca para revisión, en vez de asumir cuál era el valor correcto.

In [152]:
for c in ['wet_length_cm_raw', 'wet_wood_weight_g', 'dry_wood_weight_g', 'crust_thickness_cm',
          'leaf_dry_weight_g', 'leaf_fresh_weight_g', 'leaf_area_cm2']:
     unificado[c] = unificado[c].apply(a_numero)
     unificado['wet_length_cm'] = unificado['wet_length_cm_raw']

grosor_cols = ['leaf1_thickness1', 'leaf1_thickness2', 'leaf1_thickness3',
               'leaf2_thickness1', 'leaf2_thickness2', 'leaf2_thickness3',
               'leaf3_thickness1', 'leaf3_thickness2', 'leaf3_thickness3']
for c in grosor_cols:
    if c in unificado.columns:
        malo = unificado[c].notna() & (unificado[c] > 2)
        if malo.any():
            unificado.loc[malo, 'QC_flag'] += (
                f'{c}={unificado.loc[malo, c].tolist()} fuera de rango fisico '
                '(posible error de decimal, ej. 17.00 en vez de 0.17), excluido del promedio; ')
            unificado.loc[malo, c] = np.nan

unificado['mean_leaf_thickness_mm'] = unificado[[c for c in grosor_cols if c in unificado.columns]].mean(axis=1)

## 9. Calcular los rasgos funcionales

Mismas fórmulas usadas en Sumaco y Galeras (barrenador de 0.5 cm de diámetro → radio 0.25 cm). Como aquí sí hay peso fresco de hoja, se agrega **LDMC**:

$$LDMC\ (mg/g) = \frac{peso\ seco\ hoja\ (g) \times 1000}{peso\ fresco\ hoja\ (g)}$$

`force to punch` sigue sin poder calcularse (no hay fuerza de punzón cruda en ninguna de las 2 bases).

In [153]:
radio_barrenador_cm = 0.5 / 2  # 0.25 cm

unificado['wood_volume_cm3'] = np.pi * radio_barrenador_cm**2 * unificado['wet_length_cm']
unificado['wood_density_g_cm3'] = unificado['dry_wood_weight_g'] / unificado['wood_volume_cm3']
unificado['WSG'] = unificado['wood_density_g_cm3'] / 1.0

unificado['stem_water_content_pct'] = (
    (unificado['wet_wood_weight_g'] - unificado['dry_wood_weight_g']) / unificado['dry_wood_weight_g'] * 100
)

unificado['SLA_cm2_g'] = unificado['leaf_area_cm2'] / unificado['leaf_dry_weight_g']
unificado['LDMC_mg_g'] = unificado['leaf_dry_weight_g'] * 1000 / unificado['leaf_fresh_weight_g']
unificado['force_to_punch_kN_m'] = np.nan

unificado[['wood_density_g_cm3', 'WSG', 'stem_water_content_pct',
           'mean_leaf_thickness_mm', 'SLA_cm2_g', 'LDMC_mg_g']].describe().T

,count,mean,std,min,25%,50%,75%,max
wood_density_g_cm3,57.0,0.656098,0.126286,0.387848,0.561383,0.658167,0.752872,0.903589
WSG,57.0,0.656098,0.126286,0.387848,0.561383,0.658167,0.752872,0.903589
stem_water_content_pct,57.0,89.977203,27.550875,41.666667,70.731707,86.486486,102.666667,177.586207
mean_leaf_thickness_mm,61.0,0.176230,0.050395,0.080000,0.138889,0.170000,0.207778,0.325556
SLA_cm2_g,59.0,115.672006,38.888146,63.001396,94.018973,107.967484,136.221946,308.054286
LDMC_mg_g,59.0,417.468617,144.684878,209.655638,352.768987,405.405405,456.441862,1310.551559


## 10. Revisar rangos y consistencia lógica

Además de los rangos usados en Sumaco/Galeras, se chequea que el peso seco de hoja nunca supere al fresco (regla física básica).

In [154]:
m = unificado['leaf_dry_weight_g'] > unificado['leaf_fresh_weight_g']
unificado.loc[m, 'QC_flag'] += 'peso seco de hoja > peso fresco (imposible); '

m = unificado['dry_wood_weight_g'] > unificado['wet_wood_weight_g']
unificado.loc[m, 'QC_flag'] += 'peso seco de madera > peso humedo; '

rangos = {
    'wood_density_g_cm3': (0.15, 1.2),
    'WSG': (0.15, 1.2),
    'stem_water_content_pct': (30, 250),
    'mean_leaf_thickness_mm': (0.05, 1.0),
    'SLA_cm2_g': (20, 500),
    'LDMC_mg_g': (100, 600),
}
for col, (lo, hi) in rangos.items():
    fuera = unificado[col].notna() & ((unificado[col] < lo) | (unificado[col] > hi))
    unificado.loc[fuera, 'QC_flag'] += f'{col} fuera de rango [{lo}-{hi}]; '

sin_taxo = unificado['family'].isna() & unificado['genus'].isna() & unificado['species'].isna()
unificado.loc[sin_taxo, 'QC_flag'] += 'sin familia/genero/especie; '

sin_id = unificado['treeID'].isna() & unificado['new_tree_ID_2025'].isna()
unificado.loc[sin_id, 'QC_flag'] += 'sin treeID ni new_tree_ID_2025; '

print('Filas marcadas:', (unificado['QC_flag'] != '').sum(), 'de', len(unificado))

Filas marcadas: 11 de 62


## 11. Reordenar columnas

In [155]:
front = ['Site', 'PlotID', 'Subplot', 'treeID', 'new_tree_ID_2025',
         'family', 'genus', 'species', 'sampling_date', 'altitude_m',
         'n_leaves', 'leaf_fresh_weight_g', 'leaf_dry_weight_g', 'leaf_area_cm2',
         'SLA_cm2_g', 'SLA_original', 'LDMC_mg_g', 'mean_leaf_thickness_mm',
         'leaf1_thickness1', 'leaf1_thickness2', 'leaf1_thickness3',
         'leaf2_thickness1', 'leaf2_thickness2', 'leaf2_thickness3',
         'leaf3_thickness1', 'leaf3_thickness2', 'leaf3_thickness3',
         'wood_characteristics', 'crust_thickness_cm', 'wet_length_cm_raw', 'wet_length_cm',
         'wet_wood_weight_g', 'dry_wood_weight_g', 'wood_volume_cm3',
         'wood_density_g_cm3', 'WSG', 'stem_water_content_pct', 'force_to_punch_kN_m',
         'date_census1', 'dbh_census1', 'date_census2', 'dbh_census2', 'height_census2_m',
         'compound_leaf_note', 'nota_revision_campo', 'comment', 'comment_dinamica', 'comment_wood',
         'JH_herbarium', 'QC_flag', 'source_file', '_orig_row']
otras = [c for c in unificado.columns if c not in front]
unificado = unificado[front + otras]
unificado.shape

(62, 60)

## 12. Guardar y volver a aplicar el resaltado amarillo

Igual que en Galeras: se guarda el Excel con `pandas` y se reabre con `openpyxl` para repintar de amarillo las celdas que lo estaban en `04` (la hoja `Rasgos` de `v2` no tenía resaltado real que preservar).

In [156]:
out_path = 'Rasgos_SelvaViva_unificado.xlsx'
reporte = unificado[unificado['QC_flag'] != ''][
    ['Site', 'PlotID', 'treeID', 'new_tree_ID_2025', 'family', 'genus', 'species', 'QC_flag', 'source_file']
]

with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    unificado.to_excel(writer, sheet_name='Rasgos_SelvaViva_unificado', index=False)
    reporte.to_excel(writer, sheet_name='Filas_a_revisar', index=False)

wb = openpyxl.load_workbook(out_path)
ws = wb['Rasgos_SelvaViva_unificado']
headers = [c.value for c in next(ws.iter_rows(min_row=1, max_row=1))]
col_idx = {h: i + 1 for i, h in enumerate(headers)}
amarillo = PatternFill(start_color='FFFFFF00', end_color='FFFFFF00', fill_type='solid')

for i, row in unificado.reset_index(drop=True).iterrows():
    origen = row['source_file']
    orig_row = row['_orig_row']
    resaltado_orig = resaltado_04 if origen.startswith('04') else resaltado_v2
    ren_dict = ren04 if origen.startswith('04') else renv2
    cols_resaltadas = resaltado_orig.get(orig_row, set())
    for orig_col in cols_resaltadas:
        nuevo_col = ren_dict.get(orig_col, orig_col)
        if nuevo_col in col_idx:
            ws.cell(row=i + 2, column=col_idx[nuevo_col]).fill = amarillo

wb.save(out_path)
print('Guardado con resaltado:', out_path)

Guardado con resaltado: Rasgos_SelvaViva_unificado.xlsx


## 13. Resumen y sugerencias de revisión manual

- **1 árbol muestreado en ambas campañas** (`new_tree_ID_2025 = 3658`, plot SEV_70): compara sus valores de peso seco de hoja/madera entre feb-2025 y sept-2025 — un cambio grande podría indicar un error de captura o un cambio real de la hoja entre fechas.
- **7 notas de campo ya existentes** ("Revisar Jurgen", "revisar campo", etc., trasladadas al `QC_flag`): decidir con Jurgen/Karina si esos árboles se mantienen en la base de rasgos o se excluyen.
- **1 fila con peso seco de hoja mayor al fresco** (plot SEV_70, new ID 3642) — dato físicamente imposible, revisar la báscula/captura original.
- **1 celda de grosor de hoja con `17.00`** (plot SEV_74, árbol 8839) — con casi total seguridad falta el punto decimal (`0.17`); confirmar y corregir en el archivo fuente.
- **`force to punch` sigue sin poder calcularse**: no hay fuerza de punzón cruda en ninguna de las 2 bases de Selva Viva (a diferencia de `LDMC`, que aquí sí se pudo calcular gracias al peso fresco de hoja).
- El resaltado amarillo original de `04` se conservó en el archivo final; recuerda confirmar con quien lo marcó qué significaba antes de asumir que es siempre "dato dudoso".